In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import scrublet as scr
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import rcParams
from matplotlib import rc
from matplotlib import colors
import seaborn as sb
from collections import Counter
import glob

sc.settings.set_figure_params(dpi = 160, color_map = 'RdPu', dpi_save = 300, format = 'pdf')
plt.rcParams["figure.figsize"] = [6,6]

In [2]:
sb.set_palette('colorblind')

In [5]:
# set working directory
%cd /nfs/team205/cs42/HSC/HSC_new_to_transfer/

/nfs/team205/cs42/HSC/HSC_new_to_transfer


In [6]:
from matplotlib import font_manager
font_manager.fontManager.addfont("csv/Helvetica.ttf")
print(font_manager.findfont("Helvetica"))
plt.rcParams["font.sans-serif"] = ["Helvetica"]
plt.rcParams["pdf.fonttype"] = 42

plt.rcParams['savefig.dpi'] = 300

plt.rcParams["axes.grid"] = False
plt.rcParams["pdf.fonttype"] = 42

/nfs/team205/cs42/HSC/HSC_new_to_transfer/csv/Helvetica.ttf


In [7]:
# figure directory
fig_path = 'figure/'
sc.settings.figdir = fig_path

# Download dataset

In [8]:
smp_dict = ['wild-type','NR4A3','NR4A2','SMAD7']

In [9]:
# check whether all files have been downloaded in matrix folder
DATAFOLDER = '/nfs/team205/cs42/HSC_matrix/'
exist = [x.split('/')[-1] for x in glob.glob(DATAFOLDER+'*')]

if len([x for x in smp_dict if x not in exist])>0:
    print([smp_dict[x][2] for x in smp_dict if x not in exist],'not in matrix folder')

In [10]:
def read_files(filename, raw_file_path = '/nfs/team205/cs42/HSC_matrix/',
               min_n_count = 1000, min_n_gene = 500, max_ne_gene = 7000):

    
    path = '%s/%s/'%(raw_file_path,filename)
    adata = sc.read_10x_h5(path+'sample_filtered_feature_bc_matrix.h5', gex_only=False)
    
    adata.obs_names = [filename+"-"+x.strip("-1") for x in adata.obs_names]
   
    # define mito
    mito_genes = np.array(adata.var_names.str.startswith('MT-')) 
    
    # caculate n_counts / n_genes per cell
    adata.obs['n_counts'] = np.sum(adata.X, axis=1).A1
    adata.obs['n_genes'] = np.sum(adata.X>0,axis=1)
    
    # calculate mito%
    adata.obs['mito'] = (np.sum(adata.X[:, mito_genes],axis=1).A1) / (np.sum(adata.X,axis=1).A1)
    
    
    # filter cells
    clist = []
    clist.append(np.array(adata.obs['n_counts'] > 1000))
    clist.append(np.array(adata.obs['n_genes'] > 500))
    #clist.append(np.array(adata.obs['n_genes'] < 7000))
    clist.append(np.array(adata.obs['mito'] < 0.2)) # filter out mito >= 0.2
    
    c = np.column_stack(clist).all(axis=1)
    adata = adata[c].copy()
    
    # update manifest onto .obs
    adata.obs['sample'] = filename
    
    # calculate doublets using scrublet
    #scrub = scr.Scrublet(adata.X)
    #doublet_scores, predicted_doublets = scrub.scrub_doublets(verbose=False)
    #adata.obs['doublet_scores'] = doublet_scores
    #adata.obs['predicted_doublets'] = predicted_doublets
    
    sc.write('%s_filtered'%(filename),adata)

In [11]:
from multiprocessing import Pool

with Pool(10) as p:
    p.map(read_files, [x for x in smp_dict])

/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/sit

# Merge dataset

In [12]:
ad = {}

In [13]:
for filename in smp_dict:
    ad[filename] = sc.read('%s_filtered'%(filename))
    ad[filename].var_names_make_unique()

/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [14]:
merged = None;
for key in ad: 
    if merged is None:
        merged = ad[key]
    else:
        merged = merged.concatenate(ad[key], join='outer')

/tmp/ipykernel_409478/1590458944.py:6: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merged = merged.concatenate(ad[key], join='outer')
/tmp/ipykernel_409478/1590458944.py:6: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merged = merged.concatenate(ad[key], join='outer')
/tmp/ipykernel_409478/1590458944.py:6: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merged = merged.concatenate(ad[key], join='outer')


In [15]:
Counter(merged.obs['sample'])

Counter({'SMAD7': 6120, 'NR4A3': 5886, 'NR4A2': 4611, 'wild-type': 4494})

Remove doublets

In [16]:
scrub = scr.Scrublet(merged.X)
doublet_scores, predicted_doublets = scrub.scrub_doublets(verbose=False)
merged.obs['doublet_scores'] = doublet_scores
merged.obs['predicted_doublets'] = predicted_doublets

In [18]:
merged = merged[merged.obs['doublet_scores']<0.4]
merged = merged[merged.obs['predicted_doublets']==False]

In [19]:
Counter(merged.obs['sample'])

Counter({'SMAD7': 6058, 'NR4A3': 5865, 'NR4A2': 4580, 'wild-type': 4447})

In [20]:
merged.write_h5ad('write/validation_flex_entire_data_raw_count_v2.h5ad')

## Preprocessing - normalise and log

In [22]:
merged = sc.read('write/validation_flex_entire_data_raw_count_v2.h5ad')

In [23]:
sc.pp.normalize_per_cell(merged,counts_per_cell_after=1e4,key_n_counts='n_counts')

/tmp/ipykernel_409478/4036594748.py:1: FutureWarning: Use sc.pp.normalize_total instead
  sc.pp.normalize_per_cell(merged,counts_per_cell_after=1e4,key_n_counts='n_counts')
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:591: FutureWarning: Use sc.pp.normalize_total instead
  normalize_per_cell(


In [24]:
sc.pp.log1p(merged)

In [25]:
merged.write_h5ad('write/validation_flex_entire_data_normalised_log_v2.h5ad')